# SVM lineal — Clasificación de flores Iris

**Propósito:** comprender geométricamente un SVM lineal y evaluar su capacidad de clasificar las tres especies Iris a partir de dos características. Se implementa un ejemplo basado en **sépalos** y un experimento comparativo con **pétalos**.

**Resultados de aprendizaje:** (1) identificar hiperplanos, márgenes y vectores de soporte; (2) evitar fuga de información en el escalamiento; (3) interpretar métricas por clase; (4) explorar el parámetro `C`; (5) comparar representaciones de datos.

**Secuencia:** exploración → geometría → partición y escalamiento → entrenamiento → evaluación → fronteras → vectores de soporte → regularización → sépalos frente a pétalos → conclusiones.

> Solo se estudian kernels lineales en este cuaderno. Las figuras conceptuales binarias se distinguen del clasificador multiclase real.

## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay, confusion_matrix)

RANDOM_STATE = 42
plt.rcParams.update({'figure.figsize': (8, 5), 'axes.grid': True})
print('Bibliotecas cargadas correctamente')

## 2. Cargar y explorar Iris
Iris tiene 150 flores de tres especies: **setosa**, **versicolor** y **virginica**. El dataset incluye cuatro características (longitud y ancho de sépalo, longitud y ancho de pétalo). En el ejemplo principal se utiliza **solo las dos variables de sépalo** para poder visualizar las fronteras en 2D.

In [ ]:
iris = load_iris()
X = iris.data[:, :2].copy()  # sépalos: longitud y anchura
y = iris.target.copy()
feature_names = iris.feature_names[:2]
class_names = iris.target_names
print('X:', X.shape, '| y:', y.shape)
print('Variables:', feature_names)
print('Clases:', dict(zip(class_names, np.bincount(y))))
display(pd.DataFrame(X, columns=feature_names).assign(especie=class_names[y]).head())

In [ ]:
fig, ax = plt.subplots()
for k, nombre in enumerate(class_names):
    mask = y == k
    ax.scatter(X[mask, 0], X[mask, 1], label=nombre, alpha=.75, edgecolors='k', linewidths=.4)
ax.set(xlabel=feature_names[0], ylabel=feature_names[1], title='Iris: distribución por especie (sépalos)')
ax.legend(); plt.show()

**Pregunta de observación:** ¿qué especies se solapan más? ¿Es posible separarlas perfectamente mediante rectas?

## 3. Fundamento geométrico del SVM lineal
En **clasificación binaria**, la frontera es el hiperplano $w^T x+b=0$. Las rectas de margen canónicas son $w^T x+b=+1$ y $w^T x+b=-1$; su separación es $2/\lVert w\rVert$.

Con margen blando, SVM minimiza $\frac12\lVert w\rVert^2+C\sum_i\xi_i$ sujeto a $y_i(w^Tx_i+b)\geq1-\xi_i$, $\xi_i\geq0$ (etiquetas binarias $y_i\in\{-1,+1\}$). Un **C bajo** tolera más violaciones; uno **alto** las penaliza más.

Los **vectores de soporte** son muestras de entrenamiento con influencia en la solución. Para **tres clases**, `SVC` entrena internamente clasificadores **uno contra uno (OvO)**; por ello no existe un único hiperplano y un único par de márgenes que describan todo el modelo Iris.

## 4. Partición estratificada y estandarización
Se usa 70 % de entrenamiento y 30 % de prueba. La particion preserva las proporciones de clases (stratify=y). **Se ajusta el escalador únicamente con entrenamiento (train)** y se transforma prueba (test) con el mismo escalador para evitar fuga de información.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Entrenamiento:', X_train.shape, np.bincount(y_train))
print('Prueba:', X_test.shape, np.bincount(y_test))
print('Medias de entrenamiento:', np.round(scaler.mean_, 3))

Resultado 1. Entrenamiento: (105, 2)
Significa que el conjunto de entrenamiento contiene:
- 105 observaciones (70 % de las 150 flores).
- 2 características por observación: longitud y anchura del sépalo, en centímetros.

Resultado 2. [35 35 35]
Indica que las 105 observaciones se distribuyen equitativamente entre las tres especies.

## 5. Entrenar SVM lineal
`kernel="linear"` crea fronteras lineales en el espacio escalado; `C=1.0` es un valor inicial, **no necesariamente el mejor**.

In [ ]:
model = SVC(kernel='linear', C=1.0)
model.fit(X_train_scaled, y_train)
print('Vectores de soporte por clase:', dict(zip(class_names, model.n_support_)))
print('Total de vectores de soporte:', len(model.support_))
print('Clasificadores binarios internos (OvO):', len(class_names)*(len(class_names)-1)//2)

1. Sobre vectores de soporte.
- Especie,	Muestras de entrenamiento,	Vectores de soporte.	Porcentaje.
- Setosa	35	5	14,3 %
- Versicolor	35	24	68,6 %
- Virginica	35	22	62,9 %
- Total	105	51	48,6 %

2. ¿Qué significa tener tres clasificadores binarios internos?
- SVC de Scikit-learn utiliza internamente la estrategia One-vs-One (OvO) para resolver la clasificación multiclase.
- Clasificador,	Clases que distingue
- SVM1	Setosa vs. Versicolor
- SVM2	Setosa vs. Virginica
- SVM3	Versicolor vs. Virginica

## 6. Predecir y evaluar
En este cuaderno solo se utiliza la metrica **exactitud** que describe el error del sistema SVM. Comparamos predicciones y etiquetas verdaderas; luego calculamos exactitud.

In [ ]:
y_pred = model.predict(X_test_scaled)
print('Primeras 10 predicciones:', class_names[y_pred[:10]])
print('Primeras 10 etiquetas reales:', class_names[y_test[:10]])
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')

## 7. Visualizar regiones de decisión y datos de entrenamiento/prueba
La malla se construye en **centímetros originales** y se escala antes de pasar al clasificador. Los puntos de entrenamiento son círculos; los de prueba, cruces. Las regiones muestran la clase predicha, no un margen único del SVM multiclase.

In [ ]:
from matplotlib.colors import ListedColormap

def plot_decision_regions(model, scaler, X_train, y_train, X_test, y_test, names, title):
    X_all = np.vstack([X_train, X_test])
    pad = 0.55
    x0_min, x0_max = X_all[:,0].min()-pad, X_all[:,0].max()+pad
    x1_min, x1_max = X_all[:,1].min()-pad, X_all[:,1].max()+pad
    xx, yy = np.meshgrid(np.linspace(x0_min, x0_max, 240),
                         np.linspace(x1_min, x1_max, 240))
    grid = np.c_[xx.ravel(), yy.ravel()]
    pred = model.predict(scaler.transform(grid)).reshape(xx.shape)
    cmap = ListedColormap(['#bfe6ef', '#f8d7a3', '#d9c5ed'])
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.contourf(xx, yy, pred, levels=[-.5,.5,1.5,2.5], cmap=cmap, alpha=.6)
    for k, name in enumerate(names):
        tr = y_train == k
        te = y_test == k
        ax.scatter(X_train[tr,0], X_train[tr,1], c=[plt.cm.tab10(k)],
                   marker='o', edgecolors='k', s=45, label=f'{name} (train)')
        ax.scatter(X_test[te,0], X_test[te,1], c=[plt.cm.tab10(k)],
                   marker='x', s=70, linewidths=1.6, label=f'{name} (test)')
    ax.set(xlabel=feature_names[0], ylabel=feature_names[1], title=title)
    ax.legend(ncol=2, fontsize=8, loc='best'); plt.show()

plot_decision_regions(model, scaler, X_train, y_train, X_test, y_test,
                      class_names, 'SVM lineal: regiones de decisión (sépalos)')

## 8. Visualizar correctamente los vectores de soporte
`model.support_vectors_` está en **coordenadas escaladas**. Para dibujarlos sobre los datos originales en centímetros es indispensable aplicar `scaler.inverse_transform`. Se muestran únicamente muestras de entrenamiento.

In [ ]:
support_vectors_original = scaler.inverse_transform(model.support_vectors_)
fig, ax = plt.subplots(figsize=(9, 6))
for k, name in enumerate(class_names):
    m = y_train == k
    ax.scatter(X_train[m, 0], X_train[m, 1], color=plt.cm.tab10(k),
               edgecolors='k', linewidths=.5, alpha=.8, label=name)
ax.scatter(support_vectors_original[:,0], support_vectors_original[:,1],
           s=150, facecolors='none', edgecolors='red', linewidths=1.8,
           label='Vectores de soporte')
ax.set(xlabel=feature_names[0], ylabel=feature_names[1],
       title='Vectores de soporte (coordenadas originales)')
ax.legend(); plt.show()
print('Índices relativos al conjunto de entrenamiento:', model.support_)
print('Número por clase:', model.n_support_)

**Nota técnica:** en un SVM multiclase, los vectores de soporte son el conjunto de muestras usadas por los clasificadores binarios internos. No deben interpretarse como puntos de un solo margen global.

## 9. Experimentar con el parámetro C
Mantenemos la misma partición y el mismo escalamiento para que la comparación sea justa. Las métricas sobre prueba son descriptivas; para seleccionar hiperparámetros rigurosamente se recomienda validación cruzada **solo en entrenamiento**.

In [ ]:
resultados_c = []
for c in [0.01, 0.1, 1, 10, 100]:
    m = SVC(kernel='linear', C=c).fit(X_train_scaled, y_train)
    resultados_c.append({'C': c,
                         'Accuracy train': accuracy_score(y_train, m.predict(X_train_scaled)),
                         'Accuracy test': accuracy_score(y_test, m.predict(X_test_scaled)),
                         'N. vectores soporte': len(m.support_)})
df_c = pd.DataFrame(resultados_c)
display(df_c.round(3))
fig, ax = plt.subplots()
ax.semilogx(df_c['C'], df_c['Accuracy train'], 'o-', label='Entrenamiento')
ax.semilogx(df_c['C'], df_c['Accuracy test'], 's-', label='Prueba')
ax.set(xlabel='C (escala logarítmica)', ylabel='Accuracy', title='Sensibilidad a C')
ax.legend(); plt.show()

### Validación cruzada para elegir C sin usar el conjunto de prueba
Se usa `Pipeline` para ajustar el escalamiento dentro de cada fold. El conjunto de prueba se reserva para la evaluación final del valor elegido.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for c in [0.01, 0.1, 1, 10, 100]:
    pipe = make_pipeline(StandardScaler(), SVC(kernel='linear', C=c))
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    cv_rows.append({'C': c, 'Accuracy CV media': scores.mean(), 'Desv. CV': scores.std()})
cv_table = pd.DataFrame(cv_rows)
display(cv_table.round(3))
best_c = float(cv_table.sort_values(['Accuracy CV media', 'C'], ascending=[False, True]).iloc[0]['C'])
final_model = make_pipeline(StandardScaler(), SVC(kernel='linear', C=best_c))
final_model.fit(X_train, y_train)
print('C elegido por CV:', best_c)
print('Accuracy final en prueba:', round(final_model.score(X_test, y_test), 3))

## 10. Comparar sépalos y pétalos
Repetimos el protocolo con dos conjuntos de variables, manteniendo **los mismos índices de entrenamiento y prueba**. Para una comparación docente sencilla usamos `C=1` en ambos; no cambiamos el kernel.

In [ ]:
indices = np.arange(len(y))
idx_train, idx_test = train_test_split(indices, test_size=.30,
                                      random_state=RANDOM_STATE, stratify=y)
comparacion = []
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (nombre, columnas) in zip(axes, [('Sépalos', [0,1]), ('Pétalos', [2,3])]):
    Xi = iris.data[:, columnas]
    pipe = make_pipeline(StandardScaler(), SVC(kernel='linear', C=1.0))
    pipe.fit(Xi[idx_train], y[idx_train])
    acc = pipe.score(Xi[idx_test], y[idx_test])
    comparacion.append({'Variables': nombre, 'Accuracy prueba': acc})
    for k, especie in enumerate(class_names):
        m = y == k
        ax.scatter(Xi[m,0], Xi[m,1], label=especie, alpha=.75, s=26)
    ax.set(xlabel=iris.feature_names[columnas[0]], ylabel=iris.feature_names[columnas[1]],
           title=f'{nombre} — Accuracy test: {acc:.3f}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
display(pd.DataFrame(comparacion).round(3))

**Discusión:** ¿qué variables ofrecen mayor separación visual? ¿La mejora en Accuracy puede atribuirse exclusivamente al algoritmo? ¿Qué limitaciones tiene una única partición de entrenamiento/prueba?

## 11. Preguntas de comprensión y actividades
1. Explique por qué un SVM lineal necesita estandarizar características con escalas diferentes.
2. Distinga hiperplano, margen y vector de soporte en el caso binario.
3. ¿Por qué un SVM multiclase OvO no posee una sola recta de separación?
4. Interprete la matriz de confusión e identifique las clases más difíciles.
5. ¿Cómo cambia el número de vectores de soporte cuando cambia `C`? Use los resultados, no una regla absoluta.
6. ¿Qué sucede si se grafican `support_vectors_` sin deshacer la estandarización?
7. Compare sépalos y pétalos con la misma partición. Justifique la diferencia observada.
8. ¿Por qué el escalador debe entrenarse dentro de cada fold de validación cruzada?


## 12. Síntesis
- SVM lineal busca separar clases mediante fronteras lineales con regularización.
- En Iris multiclase, `SVC` utiliza clasificadores binarios uno contra uno.
- La escala de los datos, el valor de `C` y las características elegidas influyen en los resultados.
- La interpretación requiere métricas por clase y una evaluación sin fuga de información.

**Extensión futura:** kernels no lineales, fuera del alcance de esta práctica.